# Food Category Classification — Transfer Learning

Trains a ViT model on 8 food categories:
**Chicken, Bread, Meat, Soup, Salad, Dessert, Seafood, Pasta**

Base model: `google/vit-base-patch16-224`
Dataset: `juliaper/food-cv-dataset`

In [ ]:
import sys
!{sys.executable} -m pip install 'accelerate>=1.1.0' 'transformers[torch]' datasets evaluate huggingface_hub Pillow --upgrade -q

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor, ViTForImageClassification, Trainer, TrainingArguments
import evaluate

print('GPU available:', torch.cuda.is_available())

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Load Dataset

The dataset is structured as `food_cv_data/train/ClassName/` and `food_cv_data/test/ClassName/`

In [ ]:
# Load dataset — images are in food_cv_data/train and food_cv_data/test subfolders
dataset = load_dataset(
    'juliaper/food-cv-dataset',
    data_dir='food_cv_data'
)
print(dataset)
labels = dataset['train'].features['label'].names
print('Classes:', labels)

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

In [ ]:
model_name = 'google/vit-base-patch16-224'
image_processor = AutoImageProcessor.from_pretrained(model_name)

def preprocess(batch):
    images = [img.convert('RGB') for img in batch['image']]
    inputs = image_processor(images, return_tensors='pt')
    inputs['labels'] = batch['label']
    return inputs

dataset = dataset.map(preprocess, batched=True, remove_columns=['image'])
dataset.set_format('torch')
print('Preprocessing done.')
print(dataset)

In [ ]:
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
print('Model loaded. Classes:', len(labels))

In [ ]:
accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
training_args = TrainingArguments(
    output_dir='vit-food-category',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    push_to_hub=True,
    hub_model_id='juliaper/vit-food-category',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()

In [ ]:
# Push model and image processor to Hugging Face
trainer.push_to_hub()

image_processor_final = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
image_processor_final.push_to_hub('juliaper/vit-food-category')

print('Done! Model at: https://huggingface.co/juliaper/vit-food-category')